# പാഠം 12 - ഏജന്റ് സ്‌ക്രാച്‌പാഡ് ഉപയോഗിച്ച് चैറ്റ് ചരിത്ര കുറവ്

ഈ നോട്ട്ബുക്ക് Microsoft Agent Framework ഉപയോഗിച്ച് നീണ്ട സംഭാഷണങ്ങളിലെ കോണ്ടെക്സ്റ്റ് എങ്ങനെ നിയന്ത്രിക്കാമെന്ന് കാണിക്കുന്നു. സംഭാഷണങ്ങൾ മൂട്ടുമ്പോൾ ടോക്കൺ എണ്ണവും വർധിക്കുന്നു — അവസാനം മോഡലിന്റെ കോണ്ടെക്സ്റ്റ് വിൻഡോയിലുപരി. ഇത് നമ്മൾ **കോണ്ടെക്സ്റ്റ് സംഗ്രഹീകരണ മാതൃക**യും ശാശ്വത മെയ്മറിയായി പ്രവർത്തിക്കുന്ന **ഏജന്റ് സ്‌ക്രാച്‌പാഡ്**ഉം ഉപയോഗിച്ചാണ് കൈകാര്യം ചെയ്യുന്നത്.

## നിങ്ങൾ പഠിക്കാനിരിക്കുന്നതാകുന്നു:
1. **കോണ്ടെക്സ്റ്റ് മാനേജ്മെന്റിന്റെ പ്രാധാന്യം**: ടോക്കൺ പരിധികളും കോണ്ടെക്സ്റ്റ് വിൻഡോസും മനസിലാക്കൽ
2. **കോണ്ടെക്സ്റ്റ്-അവേർ ഏജന്റുകൾ**: സ്വന്തം സംഭാഷണ കോണ്ടെക്സ്റ്റ് ചെയതെടുക്കുന്ന ഏജന്റുകൾ നിർമ്മിക്കൽ
3. **കോണ്ടെക്സ്റ്റ് സംഗ്രഹീകരണ മാതൃക**: സംഭാഷണ ചരിത്രം സംക്ഷിപ്തമാക്കാനുളള ടൂൾസ് ഉപയോഗിക്കൽ
4. **ഏജന്റ് സ്‌ക്രാച്‌പാഡ്**: കോണ്ടെക്സ്റ്റ് കുറവിനെ അതിക്രമിച്ച് നിലനിൽക്കുന്ന ശാശ്വത മെയ്മറി

## മുൻപരിചയങ്ങൾ:
- പരിസ്ഥിതി വ്യത്യാസങ്ങൾ ക്രമീകരിച്ച Azure OpenAI സെറ്റപ്പ്
- മുമ്പത്തെ പാഠങ്ങളിൽ നിന്നുള്ള അടിസ്ഥാന ഏജന്റ് ആശയങ്ങളുടെ മനസ്സിലാക്കൽ


## സംവിധാനം


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity --quiet

In [ ]:
import os
import asyncio
from datetime import datetime
from pathlib import Path

from dotenv import load_dotenv

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
# Load environment variables and create Azure AI Foundry provider
load_dotenv()

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

print("✅ Azure AI Foundry provider configured")


## കോൺടെക്സ്റ്റ് മാനേജ്മെന്റ് എന്തിനാണ് പ്രധാനപ്പെട്ടത്

ഓരോ LLM-ക്കും ഒരെണ്ണം ഫിനിറ്റി **കോൺടെക്സ്റ്റ് വിൻഡോ** ഉണ്ട് — ഒരു അഭ്യർത്ഥനയിൽ ഒരേസമയം оноടാനാകുന്ന പരമാവധി ടോക്കൺസ്. ബഹു-തവണ സംഭാഷണം പുരോഗമിക്കുമ്പോൾ:

- **ടോക്കൺ‌സ് എണ്ണം രേഖാഭാഷയിൽ വർദ്ധിക്കുന്നു** ഓരോ ഉപയോക്തൃ സന്ദേശത്തോടും സഹായി മറുപടിയോടും.
- **പ്രോംപ്റ്റ് ടോക്കൺസ് ചെലവ് നിയന്ത്രിക്കുന്നു** കാരണം മുഴുവൻ ചരിത്രവും ഓരോ തവണയുമും വീണ്ടും അയയ്ക്കുന്നു.
- അവസാനം സംഭാഷണം **കോൺടെക്സ്റ്റ് വിൻഡോ കടന്നുതുടങ്ങും** മോഡൽ അല്ലെങ്കിൽ മുറിച്ചെടുക്കുന്നു അല്ലെങ്കിൽ പിശക് കാണിക്കുന്നു.

### കോൺടെക്സ്റ്റ് മാനേജ്മെന്റ് തന്ത്രങ്ങൾ

| തന്ത്രം | ഇത് എങ്ങനെ പ്രവർത്തിക്കുന്നു | വ്യാപാരം |
|---|---|---|
| **മുറിച്ചെടുപ്പ്** | പഴയ സന്ദേശങ്ങൾ ഒഴിവാക്കുക | മെഡൽ ആരംഭത്തിലെ കോൺടെക്സ്റ്റ് നഷ്ടപ്പെടുന്നു |
| **സംഗ്രഹം** | പഴയ സന്ദേശങ്ങൾ ഒരു സംഗ്രഹത്തിൽ ചുരുക്കുക | ചില വിശദാംശങ്ങൾ നഷ്ടപ്പെടുന്നു, പക്ഷേ പ്രധാന കാര്യങ്ങൾ നിലനിർത്തുന്നു |
| **സ്ക്രാച്ച്പാഡ് / ബാഹ്യ മെമ്മറി** | പ്രധാന വിവരങ്ങൾ സംഭാഷണത്തിന് ബാഹ്യമായി സൂക്ഷിക്കുക | ഉപകരം വിളികളിലൂടെ പ്രവർത്തനക്ഷമമാകുന്നു, പക്ഷേ വാരിയാന്നും നിലനിർത്തുന്നു |

ഇത് നോട്ട്‌ബുക്കിൽ ഞങ്ങൾ **സംഗ്രഹം** ഒരു **സ്ക്രാച്ച്പാഡ് ഉപകരണം** എന്നിവ ചേർത്ത് ഉപയോഗിക്കുന്നു, അതിലൂടെ ഏജൻറ് സംവേദനചരിത്രം ചുരുക്കിയിട്ടും തുടർച്ച നിലനിർത്താൻ സാധിക്കും.


## ഒരു സാന്ദർഭ്യ-അറിവുള്ള ഏജന്റ് സൃഷ്ടിക്കൽ


In [ ]:
agent = await provider.create_agent(
    name="ContextAwareAgent",
    instructions="""You are a helpful travel planning assistant with excellent memory management.
When conversations get long:
1. Summarize previous context into key points
2. Track user preferences mentioned earlier
3. Reference previous decisions without repeating full details
Always maintain continuity while being concise.""",
)

print("🤖 Context-aware travel planning agent created")

## ഒരു ദീർഘമായസംഭാഷണം അനുകരിക്കൽ

സന്ദർഭം എങ്ങനെ സമാഹരിക്കപ്പെടുന്നു എന്ന് കാണാൻ ഒരു ബഹുനിരസംഭാഷണം നടത്താം. ഏജൻറ് പ്രധാന വിവരങ്ങൾ (രുചികൾ, ബജറ്റ്, യാത്രാ തിയതികൾ) സംഭാഷണത്തിനിടെ ഇൻഹേറിറ്റ് ചെയ്ത് തുടർച്ച കാണിക്കmalıdır.


In [ ]:
session = agent.create_session()

# Turn 1 - Initial preferences
response = await agent.run("I'm planning a trip to Japan. I love sushi, temples, and photography.", session=session)
print(f"Turn 1: {response}\n")

# Turn 2 - More details
response = await agent.run("My budget is $3000 and I'll be traveling solo for 10 days in April.", session=session)
print(f"Turn 2: {response}\n")

# Turn 3 - Test context retention
response = await agent.run("Based on everything I've told you so far, what's the one thing you'd recommend I not miss?", session=session)
print(f"Turn 3: {response}\n")

ഏജന്റ് മുൻ സംഭാഷണങ്ങളിൽ നിന്നുള്ള സാന്ദർഭ്യം സംരക്ഷിക്കുന്നത 어떻게 ശ്രദ്ധിക്കുക — ജപ്പാൻ, സുഷി, ക്ഷേത്രങ്ങൾ, ഫോട്ടോഗ്രഫി, $3000 ബജറ്റ്, ഒറ്റയാത്ര, ഏപ്രിൽ കാലഘട്ടം എന്നിവയെക്കുറിച്ച് അതിന് അറിവുണ്ട്. ഒരു ചെറു സംഭാഷണത്തിൽ ഇത് നന്നായി പ്രവർത്തിക്കുന്നു, പക്ഷേ സംഭാഷണം വന്നെറിഞ്ഞതോടെ മുഴുവൻ ചരിത്രം വീണ്ടും അയക്കുന്നത് ചെലവേറിയ കാര്യമായിരിക്കും.

സാന്ദർഭം സഞ്ചയിക്കുന്നതിനെ കാണാൻ കൂടുതലുള്ള തവണുകൾ കൊണ്ട് സംഭാഷണം തുടരാം:


In [ ]:
# Turn 4 - Expand the conversation
response = await agent.run("What about accommodation? I prefer traditional Japanese inns.", session=session)
print(f"Turn 4: {response}\n")

# Turn 5 - Change of plans
response = await agent.run("Actually, I've changed my mind about the dates. I'll go in October instead for the autumn colors.", session=session)
print(f"Turn 5: {response}\n")

# Turn 6 - Test retention after change
response = await agent.run("Summarize my complete travel plan so far — destination, budget, duration, interests, accommodation, and timing.", session=session)
print(f"Turn 6: {response}\n")

## സന്ദർഭ സംക്ഷേപണം മാതൃക

സംഭാഷണം വളർന്ന് പോകുമ്പോൾ, ചുരുക്കിയ രൂപത്തിലേക്ക് സമാഹരിച്ച് സന്ദർഭം പുനഃസംക്ഷേപിക്കാൻ നാം ഒരു **സംക്ഷേപണ ഉപകരണം** ഉപയോഗിക്കാം. മൂന്നാം തരത്തിലുള്ള സന്ദേശങ്ങൾ ഒഴിവാക്കപ്പെട്ടാലും പ്രധാനപ്പെട്ട വിവരങ്ങൾ സംരക്ഷിക്കാൻ ഏജന്റ് ഈ ഉപകരണത്തെ കോളുചെയ്യുന്നു.

ഈ മാതൃക കൂടുതൽ സങ്കീർണ്ണമായ ചരിത്രം കുറയ്ക്കലിന്റെ അടിസ്ഥാന ഘടകമാണ്:
1. സംഭാഷണത്തിൽ നിന്നുള്ള പ്രധാന വിവരങ്ങൾ ഏജന്റ് തിരിച്ചറിയുന്നു
2. അവ പിൻവലിക്കാൻ സംക്ഷേപണ ഉപകരണത്തെ കോളുചെയ്യുന്നു
3. പഴയ സന്ദേശങ്ങൾ സുരക്ഷിതമായി നീക്കം ചെയ്യാവുന്നതാണ് കാരണം സംക്ഷേപണം പ്രധാനപ്പെട്ടവ കൈവരിക്കുന്നു

താഴെ, ഏജന്റ് മുഖേന കോളുചെയ്യാവുന്ന `summarize_preferences` എന്ന ഒരു ഉപകരണം നിർവചിച്ചിരിക്കുന്നു, ഇത് പഠിച്ചു കൊണ്ടിരിക്കുന്നവയുടെ ചുരുക്കം രേഖപ്പെടുത്തുന്നു.


In [ ]:
@tool(approval_mode="never_require")
def summarize_preferences(conversation_notes: str) -> str:
    """Summarize accumulated user preferences into a compact format."""
    return f"[SUMMARY] User preferences recorded: {conversation_notes}"


# Create an enhanced agent with the summarization tool
summarizing_agent = await provider.create_agent(
    name="SummarizingTravelAgent",
    instructions="""You are a helpful travel planning assistant that actively manages conversation context.

CONTEXT MANAGEMENT RULES:
1. After gathering several user preferences, call summarize_preferences() to record a compact summary
2. When the user asks you to recall details, reference your recorded summaries
3. Keep responses concise — avoid restating the entire history

PLANNING PROCESS:
1. Gather user preferences (destination, budget, dates, interests)
2. Summarize preferences using the tool
3. Create recommendations based on the summary
4. Update the summary when preferences change""",
    tools=[summarize_preferences],
)

print("🤖 Summarizing travel agent created with context tools")

In [ ]:
# Demonstrate the summarization pattern
summary_session = summarizing_agent.create_session()

# Provide a batch of preferences
response = await summarizing_agent.run(
    "I want to visit Greece. I love seafood, history, and island hopping. "
    "Budget is $4000 for two weeks. Traveling with my partner in June. "
    "Please record these preferences using your summarization tool.",
    session=summary_session,
)
print(f"Agent: {response}\n")

# Ask the agent to use the recorded context
response = await summarizing_agent.run(
    "Now, based on what you've recorded, suggest the top 3 islands we should visit.",
    session=summary_session,
)
print(f"Agent: {response}\n")

## സാരാംശം

ഈ പാഠത്തിൽ നിങ്ങള്‍ Microsoft Agent Framework ഉപയോഗിച്ച് ദൈർഘ്യമേറിയ ഏജന്റ് സംഭാഷണങ്ങളിൽ കോൺടെക്‌സ്‌റ്റ് എങ്ങനെയാണ് കൈകാര്യം ചെയ്യേണ്ടത് എന്ന് പഠിച്ചു:

### പ്രധാന ആശയങ്ങള്‍
- **Context windows കഴിഞ്ഞുപോകുന്നവയാണ്** — സംഭാഷണ ചരിത്രത്തിലെ ഓരോ ടോക്കണും പണം ചിലവാകും കൂടാതെ പരിധിക്കുള്ളില്‍ എണ്ണപ്പെടും.
- **സംഗ്രഹണ ഉപകരണങ്ങൾ** ഏജന്റിന് സമാഹരിച്ച കോൺടെക്‌സ്‌റ്റ് സംക്ഷിപ്തമായ സാരാംശങ്ങളാക്കി മാറ്റാൻ സഹായിക്കുന്നു, ടോക്കൺ ഉപയോഗം കുറയ്ക്കുകയും അതേസമയം അനിവാര്യമായ വിവരങ്ങൾ സംരക്ഷിക്കുകയും ചെയ്യുന്നു.
- **ഏജന്റ് സ്ക്രാച്ച്‌പാഡുകൾ** ഏതൊരു സംഭാഷണ കുറയ്ക്കലും കടന്നുപോകുന്ന സ്ഥിരമായ ബാഹ്യ സ്മരണ നൽകുന്നു.

### നിങ്ങൾ നിർമ്മിച്ചത്
- ഒറ്റപ്പെടലില്ലായ്മ ഉറപ്പുവരുത്തുന്ന **context-aware agent** multi-turn സംഭാഷണങ്ങളിലൂടെ തുടർച്ച കൈവരിക്കുന്നത്
- പ്രധാന ഉപയോക്തൃ വിവരങ്ങൾ സംക്ഷിപ്ത ഫോർമാറ്റിൽ രേഖപ്പെടുത്തുന്ന **സംഗ്രഹണ ഉപകരണം** (`summarize_preferences`)
- context നിലനിർത്തലും മാറ്റം കൈകാര്യം ചെയ്യലും പ്രദർശിപ്പിക്കുന്ന **മൾട്ടി-ടേൺ സംഭാഷണം**

### യാഥാർത്ഥ്യ പ്രയോഗങ്ങൾ
- **കസ്റ്റമർ സർവീസ് ബോട്ടുകൾ**: ദൈർഘ്യമേറിയ പിന്തുണ സെഷനുകളിലൂടെ മുൻഗണനകൾ ഓർക്കുക
- **പ്രതിഷ്ഠിത വ്യക്തിഗത സഹായികൾ**: context വീണ്ടും വിശദീകരിക്കാതെ പുരോഗമിക്കുന്ന പദ്ധതികൾ ട്രാക്ക് ചെയ്യുക
- **ശിക്ഷണ സഹായികൾ**: നിരവധി ഇടപഴകലുകൾക്കിടയിൽ വിദ്യാർത്ഥിയുടെ പുരോഗതി നിലനിർത്തുക

### അടുത്ത് ചെയ്യേണ്ട കാര്യങ്ങൾ
- ഫയൽ അടിസ്ഥാന persistence-ഉള്ള പൂർണ്ണ സ്ക്രാച്ച്‌പാഡ് ഉപകരണം നടപ്പിലാക്കുക
- സംഗ്രഹത്തിന്റെ ശേഷം സ്വയം ചരിത്രം ചെറുതാക്കി മാറ്റൽ ചേർക്കുക
- സാംവavesം സ്മരണാന്വേഷണത്തിനായി വെക്റ്റർ ഡാറ്റാബേസുകളുമായി ഓഫുക
- പൂർണ്ണ context-ഉം കൊണ്ട് ദിവസങ്ങൾക്കു ശേഷമുള്ള സംഭാഷണങ്ങൾ വീണ്ടും തുടർന്നുള്ള ഏജന്റുകൾ നിർമ്മിക്കുക


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**അസത്യവാദം**:  
ഈ ഡോക്യുമെന്റ് AI തർജ്ജമാ സേവനം [Co-op Translator](https://github.com/Azure/co-op-translator) ഉപയോഗിച്ച് തർജ്ജമ ചെയ്തതാണ്. നമുക്ക് കൃത്യതയുണ്ടാക്കാൻ ശ്രമിക്കുന്നുവെങ്കിലും, സ്വയംകരുതിയ തർജ്മകളിൽ പിശകുകൾ അല്ലെങ്കിൽ അശുദ്ധികൾ ഉണ്ടാകാമെന്നും ദയവായി ശ്രദ്ധിക്കുക. മൗലിക ഭാഷയിലുള്ള ഒറിജിനൽ ഡോക്യുമെന്റ് അതിന്റെ പ്രാമാണിക ഉറവിടമായി കണക്കാക്കി കാണേണ്ടതാണ്. നിർണ്ണായകമായ വിവരങ്ങൾക്ക് പ്രൊഫഷണൽ മനുഷ്യൻറെ തർജ്ജമ നിർദേശിക്കപ്പെടുന്നു. ഈ തർജ്ജമ ഉപയോഗിക്കുമ്പോൾ ഉണ്ടാകുന്ന തെറ്റിദ്ധാരണകൾക്കും വ്യാഖ്യാനക്കുറവുകൾക്കുമായി നാം ഉത്തരവാദിത്തമൊതുക്കുന്നില്ല.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
